In [1]:
#easier to use notebook when getting started: using this for minor bits as I expand the database
# Krista 28 August 2026

In [2]:
%reset -f

In [3]:
#there is still some cache in Jupyter notebook that is causing issues
%load_ext autoreload
try:
    %autoreload 2 #the 2 means be aggressive and always reload imported modules
except: 
    %reload_ext autoreload

In [14]:
from sqlalchemy import select, func, create_engine, text, union_all, union
from sqlalchemy.orm import sessionmaker, Session
from sqlalchemy import inspect, MetaData, Table
from sqlalchemy import or_

import pandas as pd
import matplotlib.pyplot as plt

import pdb

import models 
#from models import *
# import models as m

In [15]:
# create a SQLite database engine
SQLALCHEMY_DATABASE_URL = "sqlite:///test_data/sargasso.db"
#this will end up creating a new database everytime, but I need this for testing right now
#SQLALCHEMY_DATABASE_URL = f"sqlite:///test_data/new_database_{datetime.now().strftime('%Y%m%d_%H%M%S')}.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL) #, echo=True) #(turn off echo, gets annoying)

#this may be way to prevent the database from being cached: (I am sure this will be slower)
# engine = create_engine(SQLALCHEMY_DATABASE_URL,poolClass = NullPool)

# Create a session
Session = sessionmaker(bind=engine)
session = Session()

In [16]:
from sqlalchemy import inspect

inspector = inspect(engine)

# Get all table names
table_names = inspector.get_table_names()
print("Tables found:", table_names)

# Get columns for each table
for table_name in table_names:
    print(f"\nTable: {table_name}")
    columns = inspector.get_columns(table_name)
    for column in columns:
        print(f"  - {column['name']} ({column['type']})")

Tables found: ['LTTdeep', 'LTTs1', 'NCBIinhouse', 'NCBIonline', 'NCBIunreleased', 'compositeV1V2', 'cyverse', 'discrete', 'metabolites', 'metabolitesUntargeted', 'seqBasics', 'seqV1V2', 'seqV4_16S', 'seqV4_18S']

Table: LTTdeep
  - id (INTEGER)
  - sample (VARCHAR)
  - biosample (VARCHAR)
  - sraV1V2 (VARCHAR)
  - year (VARCHAR)
  - month (VARCHAR)
  - depth (VARCHAR)
  - bottleID (VARCHAR)

Table: LTTs1
  - id (INTEGER)
  - biosample (VARCHAR)
  - sraV1V2 (VARCHAR)
  - year (VARCHAR)
  - month (VARCHAR)
  - depth (VARCHAR)
  - bottleID (VARCHAR)

Table: NCBIinhouse
  - id (INTEGER)
  - biosample (VARCHAR)
  - cruise5 (VARCHAR)
  - sampleV1V2 (VARCHAR)
  - sraV1V2 (VARCHAR)
  - seqV1V2 (VARCHAR)
  - sampleV416s (VARCHAR)
  - sraV416s (VARCHAR)
  - seqV416s (VARCHAR)
  - firstReference (VARCHAR)
  - bottleID (VARCHAR)

Table: NCBIonline
  - id (INTEGER)
  - contact (VARCHAR)
  - biosample (VARCHAR)
  - sample (VARCHAR)
  - sra (VARCHAR)
  - date (VARCHAR)
  - depth (VARCHAR)
  - lat (VA

In [14]:
# 1. Modify the query to perform a LEFT OUTER JOIN and filter for missing records
query = (
    select(models.LTTdeep)  # Pull records from LTTdeep
    .join(
        models.NCBIinhouse,
        models.NCBIinhouse.biosample == models.LTTdeep.biosample,
        isouter=True  # 👈 This turns the join into a LEFT JOIN
    )
    .where(
        models.NCBIinhouse.biosample == None  # 👈 Filters for records NOT found in NCBIinhouse
    )
)

# 2. Safely execute the query using a connection context manager
with engine.connect() as connection:
    df_not_found = pd.read_sql_query(query, con=connection)

# 3. View the records that failed to match
print(df_not_found)

    id        sample     biosample      sraV1V2  year month depth    bottleID
0    1      35_0M_S1  SAMN52634534  SRR36513417  1991     8     0  1003500112
1    2      42_0M_S3  SAMN52634541  SRR36513416  1992     3     0  1004200312
2    3      46_0M_S5  SAMN52634545  SRR36513405  1992     7     0  1004600212
3    4     47_0M_S17  SAMN52634546  SRR36513402  1992     8     0  1004700312
4    5     56_0M_S15  SAMN52634555  SRR36513401  1993     5     0  1005600312
5    6     65_0M_S19  SAMN52634562  SRR36513400  1994     2     0  1006500601
6   10    371_0M_S27  SAMN52634710  SRR36513396  2020     8     0  1037101511
7   11    35_200M_S2  SAMN52634711  SRR36513415  1991     8   200  1003500102
8   12    42_200M_S4  SAMN52634718  SRR36513414  1992     3   200  1004200302
9   13    46_200M_S6  SAMN52634722  SRR36513413  1992     7   200  1004600202
10  14   47_200M_S18  SAMN52634723  SRR36513412  1992     8   200  1004700302
11  15   54_200M_S10  SAMN52634729  SRR36513411  1993     3   20

In [19]:
#actually, never mind, this is not necessarily a problem one biosample can have many SRAs, but one SRA should only have one biosample
# Create a subquery of just the duplicate biosample IDs
mismatched_biosamples = (
    select(NCBI_forQuery.c.biosample)
    .group_by(NCBI_forQuery.c.biosample)
    .having(func.count(NCBI_forQuery.c.sra.distinct()) > 1)
    .subquery()
)

# Fetch all rows belonging to those conflicting biosamples
detailed_conflict_query = (
    select(NCBI_forQuery)
    .where(NCBI_forQuery.c.biosample.in_(select(mismatched_biosamples.c.biosample)))
    .order_by(NCBI_forQuery.c.biosample)
)

with engine.connect() as conn:
    df = pd.read_sql_query(detailed_conflict_query,con = conn)

print(df)
df.to_excel('test_data/c.xlsx')

        id     biosample          sra
0     1366  SAMN22169531  SRR16297535
1     1411  SAMN22169531  SRS10532025
2     1590  SAMN22169532  SRR16297534
3     1410  SAMN22169532  SRS10532026
4     1758  SAMN22169533  SRR16297295
...    ...           ...          ...
2031     3  SAMN44822553  SRS23255798
2032    93  SAMN44822554  SRR31399283
2033     2  SAMN44822554  SRS23255800
2034    92  SAMN44822555  SRR31399282
2035     1  SAMN44822555  SRS23255801

[2036 rows x 3 columns]


In [ ]:
gIsR = select(models.compositeV1V2.id, 
              models.compositeV1V2.bottleID,
              models.compositeV1V2.cruise,
              models.compositeV1V2.cast,
              models.compositeV1V2.niskin)
gNotR = select(models.Discrete.id, 
              models.Discrete.bottleID,
              models.Discrete.cruise,
              models.Discrete.cast,
              models.Discrete.niskin)


discretecombined = union_all(gNotR,gIsR)

#now make the stacked table into a subquery, this is needed to we can actually query it (otherwise can only print as shown above)
discrete_forQuery = discretecombined.subquery()

In [45]:
#now, extra check, is there anything that does not match
###....and this should not be OK for LTTdeep as I am looking in V1V2...so deal with that in a minute
#last check : does the bottleID in LTTdeep match what I think it should in the discrete file?
query = (
    select(models.LTTdeep,
           discrete_forQuery.c.bottleID)   # Pull records from both tables
    .outerjoin(
        discrete_forQuery,
        models.LTTdeep.bottleID == discrete_forQuery.c.bottleID
        )
    .where(
        models.LTTdeep.bottleID == None
    )
)

# Run the query and make a dataframe
with engine.connect() as connection:
    df = pd.read_sql_query(query, con=connection)

# 3. View the records
print(df)

Empty DataFrame
Columns: [id, sample, biosample, sraV1V2, year, month, depth, bottleID, bottleID_1]
Index: []


In [35]:
#
queryDups = (
    select(NCBI_forQuery.c.biosample)
    .group_by(NCBI_forQuery.c.biosample)
    .having(func.count(NCBI_forQuery.c.biosample) > 1)
)

with engine.connect() as conn:
    df = pd.read_sql_query(queryDups,con = conn)

print(df)

         biosample
0     SAMN22169531
1     SAMN22169532
2     SAMN22169533
3     SAMN22169534
4     SAMN22169535
...            ...
1406  SAMN44822551
1407  SAMN44822552
1408  SAMN44822553
1409  SAMN44822554
1410  SAMN44822555

[1411 rows x 1 columns]


In [ ]:
# Define the SQL command to create the table and populate it simultaneously
query = (
    select(
        models.NCBIinhouse.biosample.label('biosample'),
        models.NCBIinhouse.bottleID.label('bottleID'),
        models.NCBIinhouse.sraV1V2.label('sraV1V2'),
        models.NCBIonline.biosample.label('o_biosample'),
        models.NCBIonline.sra.label('o_sra')
    )
    .join(    
        models.NCBIonline,
        models.NCBIonline.biosample == models.NCBIinhouse.biosample,
        isouter = True
    )        
    .where(
        models.NCBIinhouse.biosample == 'check discrepancy'
    )
)

df = pd.read_sql_query(query, con=engine)
df

In [ ]:
# Define the SQL command to create the table and populate it simultaneously
first_subquery = (
    select(
        models.NCBIinhouse.biosample.label('biosample'),
        models.NCBIinhouse.bottleID.label('bottleID'),
        models.NCBIinhouse.sraV1V2.label('sraV1V2'),
        models.NCBIonline.sra.label('sra_online')
    )
    .join(    
        models.NCBIonline,
        models.NCBIonline.biosample == models.NCBIinhouse.biosample,
        isouter = True
    )        
    .where(
        models.NCBIinhouse.biosample == 'check discrepancy'
    )
    .subquery()
)

second_query = (
    select(first_subquery.c.biosample, 
           first_subquery.c.bottleID,
           first_subquery.c.sra_online)
    .where(first_subquery.c.sraV1V2 == models.NCBIonline.sra)
)

df = pd.read_sql_query(second_query, con=engine)
df

In [ ]:
#reflect the tables so I can work on them
metadata_obj = MetaData()
metadata_obj.reflect(bind=engine)

test = Table('SeqInfoNCBIinhouse', metadata_obj, autoload_with=engine)
#test = Table('SeqInfoNCBIonline', metadata_obj, autoload_with=engine)
#test = Table('SeqInfoNCBIonline', metadata_obj, autoload_with=engine)

session.query(test).all()[:5] #list so head will not work
#models.SeqInfoNCBIonline() #not helpful, just gives me the columns I defined to show in models.py

In [ ]:
query = (
    select(models.SeqInfoNCBIinhouse)
    .where(models.SeqInfoNCBIinhouse.biosample.not_like('SAMN%'))
)

# Execute the query
results = session.execute(query).scalars().all()
results

In [ ]:
query = (
    select(models.SeqInfoNCBIinhouse)
    .where(models.SeqInfoNCBIinhouse.bottleID == 9161400401)
)

# Execute the query
results = session.execute(query).scalars().all()
results

In [ ]:
#Stick some code below this spot as a holding zone
raise SystemExit("Stop execution here")

In [ ]:
# Need A - B - C ...where B is the NCBIinhouse table that can link 
# (A) discrete 
# (B) NCBIinhouse table
# (C) NCBIonline
query = (
    select(
        models.Discrete.bottleID,
        models.Discrete.temp.label('Dtemp'),
        models.Discrete.sal.label('Dsal'),
        models.Discrete.nominalDepth.label('Ddepth'),
        models.NCBIonline.temp.label('Ntemp'),
        models.NCBIonline.sal.label('Nsal'),
        models.NCBIonline.contact,
        models.NCBIonline.depth.label('Ndepth')
           )
    .join(
        models.NCBIinhouse, 
          models.Discrete.bottleID == models.NCBIinhouse.bottleID
          )  # Link A to B
    .join(
        models.NCBIonline, 
          models.NCBIinhouse.biosample == models.NCBIonline.biosample
          )  # Link B to C
    # .where(models.NCBIonline.target_value == "Desired Value")
)

df = pd.read_sql_query(query, con=engine)
df.shape

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.ext.automap import automap_base

# 1. Reset the engine connection pool
if 'engine' in globals():
    engine.dispose()
engine = create_engine('sqlite:///test_data/sargasso.db')

# 2. CRITICAL: Initialize a completely NEW Base object to drop old classes
Base = automap_base()

# 3. Reflect from scratch
Base.prepare(autoload_with=engine)

# Your updated classes will now be freshly populated
print(Base.classes.keys()) 

Keep the block below last...useful if I need to break the link to re-generate the database

In [24]:
#will need to run this if I intend to re run populate_db.py, otherwise Jupyter notebook keeps the file open
#and the delete step in that *py file fails, KL 3 September 2026
def closeDatabaseNotebook():
    import gc
    import sys

    session.close()
    engine.dispose()

    # Clear Jupyter's hidden historical output cache (which holds old results)
    sys.modules[__name__].__dict__.pop('_', None)
    sys.modules[__name__].__dict__.pop('__', None)

    # Force immediate garbage collection to release the file lock
    gc.collect()
    
    
closeDatabaseNotebook()    

engine.dispose()
session.expire_all()
session.close()